In [ ]:
from flask import Flask, request, jsonify, render_template
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import os

app = Flask(__name__)

# 加载中文模型
def load_chinese_model():
    print("正在加载中文情绪模型...")
    model_path = "my_final_model/my_model_3"
    
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"中文模型路径不存在: {model_path}")
    
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    emotion_mapping = {
        "开心": 0, "平静": 1, "伤心": 2, "生气": 3, 
        "惊讶": 4, "疑问": 5, "厌恶": 6, "关心": 7,
    }

    emotion_emojis = {
        "开心": "😊", "平静": "😐", "伤心": "😢", "生气": "😠",
        "惊讶": "😲", "疑问": "🤔", "厌恶": "🤢", "关心": "❤️"
    }
    
    print("✅ 中文模型加载完成！")
    return model, tokenizer, emotion_mapping, emotion_emojis

# 加载英文模型
def load_english_model():
    print("正在加载英文情绪模型...")
    model_path = "my_final_model/E_model"
    
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"英文模型路径不存在: {model_path}")
    
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    emotion_mapping = {
        0: "sadness", 1: "joy", 2: "love", 3: "anger", 4: "fear", 5: "surprise"
    }

    emotion_emojis = {
        "sadness": "😢", "joy": "😊", "love": "❤️", 
        "anger": "😠", "fear": "😨", "surprise": "😲"
    }
    
    print("✅ 英文模型加载完成！")
    return model, tokenizer, emotion_mapping, emotion_emojis

# 全局加载模型
try:
    chinese_model, chinese_tokenizer, chinese_emotion_mapping, chinese_emotion_emojis = load_chinese_model()
    english_model, english_tokenizer, english_emotion_mapping, english_emotion_emojis = load_english_model()
except Exception as e:
    print(f"❌ 模型加载失败: {e}")
    exit(1)

def predict_emotion(text, model_type='chinese'):
    """预测文本情绪"""
    if model_type == 'chinese':
        model = chinese_model
        tokenizer = chinese_tokenizer
        emotion_mapping = chinese_emotion_mapping
        emotion_emojis = chinese_emotion_emojis
    else:
        model = english_model
        tokenizer = english_tokenizer
        emotion_mapping = english_emotion_mapping
        emotion_emojis = english_emotion_emojis
    
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        truncation=True, 
        padding=True, 
        max_length=128
    )
    
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
    
    probabilities = predictions[0].numpy()
    
    if model_type == 'chinese':
        # 中文模型：键是情绪名称，值是索引
        predicted_class = probabilities.argmax()
        emotion = list(emotion_mapping.keys())[predicted_class]
    else:
        # 英文模型：键是索引，值是情绪名称
        predicted_class = probabilities.argmax()
        emotion = emotion_mapping[predicted_class]
    
    confidence = probabilities[predicted_class]
    emoji = emotion_emojis.get(emotion, "❓")
    
    # 获取所有情绪的概率
    all_emotions = []
    if model_type == 'chinese':
        for i, prob in enumerate(probabilities):
            emotion_name = list(emotion_mapping.keys())[i]
            emotion_emoji = emotion_emojis.get(emotion_name, "❓")
            all_emotions.append({
                'emotion': emotion_name,
                'emoji': emotion_emoji,
                'confidence': float(prob)
            })
    else:
        for i, prob in enumerate(probabilities):
            emotion_name = emotion_mapping[i]
            emotion_emoji = emotion_emojis.get(emotion_name, "❓")
            all_emotions.append({
                'emotion': emotion_name,
                'emoji': emotion_emoji,
                'confidence': float(prob)
            })
    
    # 按置信度排序
    all_emotions.sort(key=lambda x: x['confidence'], reverse=True)
    
    return {
        'text': text,
        'model_type': model_type,
        'primary_emotion': emotion,
        'primary_emoji': emoji,
        'primary_confidence': float(confidence),
        'all_emotions': all_emotions
    }

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()
        if not data:
            return jsonify({'error': '无效的请求数据'})
            
        text = data.get('text', '').strip()
        model_type = data.get('model_type', 'chinese')
        
        if not text:
            return jsonify({'error': '请输入文本'})
        
        if len(text) > 500:
            return jsonify({'error': '文本过长，请控制在500字以内'})
        
        if model_type not in ['chinese', 'english']:
            return jsonify({'error': '无效的模型类型'})
        
        result = predict_emotion(text, model_type)
        return jsonify(result)
        
    except Exception as e:
        return jsonify({'error': f'预测失败: {str(e)}'})

if __name__ == '__main__':
    # 确保模板目录存在
    templates_dir = 'templates'
    if not os.path.exists(templates_dir):
        os.makedirs(templates_dir)
        print(f"创建模板目录: {templates_dir}")
    
    print("🚀 启动双语情绪分析应用...")
    print("📱 访问地址: http://127.0.0.1:8000")
    print("💡 按 Ctrl+C 停止服务器")
    
    app.run(
        debug=True, 
        host='127.0.0.1',
        port=8000,
        use_reloader=False
    )

正在加载中文情绪模型...
✅ 中文模型加载完成！
正在加载英文情绪模型...
✅ 英文模型加载完成！
🚀 启动双语情绪分析应用...
📱 访问地址: http://127.0.0.1:8000
💡 按 Ctrl+C 停止服务器
 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:8000
Press CTRL+C to quit
127.0.0.1 - - [29/Nov/2025 09:08:51] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [29/Nov/2025 09:08:52] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [29/Nov/2025 09:09:16] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [29/Nov/2025 09:09:31] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [29/Nov/2025 09:09:52] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [29/Nov/2025 09:10:07] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [29/Nov/2025 09:13:15] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [29/Nov/2025 10:00:39] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [29/Nov/2025 10:00:48] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [29/Nov/2025 10:00:53] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [29/Nov/2025 10:07:25] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [29/Nov/2025 11:41:15] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [29/Nov/2025 11:42:08] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [29/Nov/2025 11:42:49] "POST /predict HTTP/1.1" 20